# M1 — Dimensionamento de Frota

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Beerlink Distribuição

Maurício, da Cervejaria Beerlink (treinamento 1), passou um ano. A Beerlink cresceu — hoje abastece **8 bares** em São Paulo a partir do CD em Pinheiros.

Hoje paga frete avulso. Quer estruturar entrega própria, **alugando** caminhões refrigerados:

- Capacidade: **100 caixas / viagem**
- Aluguel: **R\$ 2.000 / semana** por caminhão (com motorista)
- Combustível: **R\$ 4 / km**
- Cada caminhão faz **1 viagem por semana** (segundas)

**Pergunta:** quantos caminhões alugar e quais bares cada um atende?

Para esse módulo simplificamos cada "rota" como uma **rota-estrela** (ida-volta direta ao bar mais distante do cluster). VRP completo fica fora do escopo.

## Setup — dois solvers lado a lado

A partir deste treinamento mostramos cada modelo nos **dois solvers**:

- **OR-Tools** (Google, open-source) — backbone padrão dos treinamentos Genoa
- **Gurobi** (Gurobi Optimization, comercial) — solver state-of-the-art em MIP, representado no Brasil pela **Genoa**

**Licenciamento Gurobi:**

| Modalidade | Limite | Quando usar |
|---|---|---|
| Free limited size (sem licença) | ≤ 2.000 vars, ≤ 2.000 constraints | Aprendizado, didática (todos os casos deste treinamento cabem) |
| Academic License (gratuita) | Ilimitado | Pós-graduação |
| WLS (Web License Service) | Ilimitado, multi-usuário | Implantação corporativa |
| Commercial | Ilimitado | Produção |

Os modelos aqui rodam todos no modo **free limited-size** (não precisa de licença), mas qualquer Gradus que já tenha licença Gurobi pode aplicar direto em escala.

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
from ortools.linear_solver import pywraplp
import pandas as pd

# Dados: bares, demanda semanal (caixas) e distância ao CD Pinheiros (km)
bares = pd.DataFrame({
    'bar':    ['Centro', 'Pinheiros', 'Vila Madalena', 'Moema', 'Tatuapé', 'Lapa', 'Itaim', 'Brooklin'],
    'd':      [55, 30, 50, 45, 60, 35, 40, 25],   # caixas/sem
    'dist':   [6.0, 0.5, 1.5, 4.2, 13.5, 5.8, 2.2, 5.0],  # km até o CD
})

Q = 100         # capacidade do caminhão (caixas)
RENT = 2000     # R$/sem por caminhão alugado
CKM = 4         # R$/km
MAX_TRUCKS = 6  # limite superior para evitar variáveis demais

print(bares)
print(f"\nDemanda total: {bares['d'].sum()} caixas/semana")
print(f"Mínimo teórico de caminhões: {-(-bares['d'].sum() // Q)} (lower bound = ceil(total/Q))")

## Modelo MILP — fleet sizing com rota-estrela

Variáveis:
- $y_j \in \{0,1\}$: caminhão $j$ alugado?
- $x_{ij} \in \{0,1\}$: bar $i$ atendido pelo caminhão $j$?
- $L_j \ge 0$: maior distância do caminhão $j$ (linearização do `max`)

Função objetivo:
$$\min\ 2000 \sum_j y_j + 8 \sum_j L_j$$

(O fator 8 = 2 · 4 vem de ida-volta × R\$/km.)

Restrições:
- cobertura: cada bar é atendido por **exatamente um** caminhão
- capacidade: a carga de cada caminhão $\le Q \cdot y_j$
- distância: $L_j \ge \text{dist}_i \cdot x_{ij}$ para todo $i, j$

In [ ]:
def solve_fleet_homogeneo(bares, Q, RENT, CKM, max_trucks):
    solver = pywraplp.Solver.CreateSolver('CBC')
    if solver is None:
        raise RuntimeError('CBC indisponível — tente SCIP')

    n = len(bares)
    J = range(max_trucks)
    I = range(n)
    d = bares['d'].tolist()
    dist = bares['dist'].tolist()

    # Variáveis
    y = [solver.IntVar(0, 1, f'y[{j}]') for j in J]
    x = [[solver.IntVar(0, 1, f'x[{i},{j}]') for j in J] for i in I]
    L = [solver.NumVar(0, solver.infinity(), f'L[{j}]') for j in J]

    # Cobertura: cada bar em exatamente 1 caminhão
    for i in I:
        solver.Add(sum(x[i][j] for j in J) == 1)

    # Capacidade
    for j in J:
        solver.Add(sum(d[i] * x[i][j] for i in I) <= Q * y[j])

    # Distância máxima do cluster (linearização do max)
    for j in J:
        for i in I:
            solver.Add(L[j] >= dist[i] * x[i][j])

    # FO: aluguel + ida-volta × R$/km
    solver.Minimize(
        RENT * sum(y[j] for j in J)
        + 2 * CKM * sum(L[j] for j in J)
    )

    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        print(f'Status: {status} (não-ótimo)')
        return None

    # Extrair solução
    rotas = []
    for j in J:
        if y[j].solution_value() > 0.5:
            atendidos = [bares['bar'][i] for i in I if x[i][j].solution_value() > 0.5]
            carga = sum(d[i] for i in I if x[i][j].solution_value() > 0.5)
            rotas.append({
                'caminhão': chr(ord('A') + j),
                'bares': ', '.join(atendidos),
                'caixas': carga,
                'maior dist (km)': L[j].solution_value(),
                'custo km (R$)': 2 * CKM * L[j].solution_value(),
            })
    return {
        'custo_total': solver.Objective().Value(),
        'rotas': pd.DataFrame(rotas),
        'n_caminhoes': sum(1 for j in J if y[j].solution_value() > 0.5),
    }

sol = solve_fleet_homogeneo(bares, Q, RENT, CKM, MAX_TRUCKS)
print(f"\nCaminhões alugados: {sol['n_caminhoes']}")
print(f"Custo total: R$ {sol['custo_total']:,.2f}")
print(f"  - aluguel : R$ {RENT * sol['n_caminhoes']:,.2f}")
print(f"  - combustível: R$ {sol['custo_total'] - RENT * sol['n_caminhoes']:,.2f}")
print()
print(sol['rotas'].to_string(index=False))

### O mesmo modelo em Gurobi (gurobipy)

Sintaxe `gurobipy` é mais "pythonic" que OR-Tools:

- `model.addVars(...)` cria dicionário ou matriz de variáveis em uma chamada
- `quicksum` (versão otimizada do `sum`) acelera a montagem
- Indexação multi-dimensional natural com `tupledict`

O mesmo modelo MILP do caso homogêneo:

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_fleet_homogeneo_gurobi(bares, Q, RENT, CKM, max_trucks):
    m = gp.Model('fleet_homogeneo')
    m.Params.OutputFlag = 0  # silencia log do solver

    n = len(bares)
    I = range(n); J = range(max_trucks)
    d = bares['d'].tolist()
    dist = bares['dist'].tolist()

    # Variáveis (note a sintaxe compacta com addVars)
    y = m.addVars(J, vtype=GRB.BINARY, name='y')
    x = m.addVars(I, J, vtype=GRB.BINARY, name='x')
    L = m.addVars(J, vtype=GRB.CONTINUOUS, name='L', lb=0)

    # Cobertura: cada bar em exatamente 1 caminhão
    m.addConstrs((gp.quicksum(x[i, j] for j in J) == 1 for i in I), name='cob')

    # Capacidade
    m.addConstrs((gp.quicksum(d[i] * x[i, j] for i in I) <= Q * y[j] for j in J), name='cap')

    # Distância máxima (linearização do max)
    m.addConstrs((L[j] >= dist[i] * x[i, j] for i in I for j in J), name='dist')

    # FO: aluguel + 2 · custo_km · L (ida-volta)
    m.setObjective(
        RENT * gp.quicksum(y[j] for j in J) + 2 * CKM * gp.quicksum(L[j] for j in J),
        GRB.MINIMIZE
    )

    m.optimize()
    if m.Status != GRB.OPTIMAL:
        return None

    rotas = []
    for j in J:
        if y[j].X > 0.5:
            atendidos = [bares['bar'][i] for i in I if x[i, j].X > 0.5]
            carga = sum(d[i] for i in I if x[i, j].X > 0.5)
            rotas.append({
                'caminhão': chr(ord('A') + j),
                'bares': ', '.join(atendidos),
                'caixas': carga,
                'maior dist (km)': L[j].X,
            })
    return {
        'custo_total': m.ObjVal,
        'rotas': pd.DataFrame(rotas),
        'n_caminhoes': sum(1 for j in J if y[j].X > 0.5),
        'tempo_solver': m.Runtime,
    }

sol_g = solve_fleet_homogeneo_gurobi(bares, Q, RENT, CKM, MAX_TRUCKS)
print(f"Custo total (Gurobi): R$ {sol_g['custo_total']:,.2f}")
print(f"Tempo Gurobi: {sol_g['tempo_solver']*1000:.1f} ms")
print()
print(sol_g['rotas'].to_string(index=False))

# Confirmação de que os dois solvers chegam ao mesmo ótimo
print(f"\n✓ Mesmo ótimo? OR-Tools R$ {sol['custo_total']:,.2f}  vs  Gurobi R$ {sol_g['custo_total']:,.2f}")

### Discussão

Compare a solução do modelo com o que a turma chutaria "a olho":

- **Tatuapé vai sozinho?** É o mais distante (13.5 km) e tem demanda alta (60 cx). Quase qualquer combinação com outro bar excede 100 caixas, e mesmo que coubesse, a rota até Tatuapé é cara — não compensa "economizar caminhão".
- **Pinheiros + Vila Madalena estão no centro do CD** — fáceis de juntar (80 cx, distância máx 1.5 km).
- **O algoritmo otimiza dois eixos simultaneamente:** clustering geográfico (reduz km) e capacidade (reduz #caminhões). Esses dois critérios às vezes conflitam — é aí que o solver brilha.

---

## Exercício de extensão: frota heterogênea + múltiplas viagens

Agora a Beerlink pode escolher entre **3 tipos de caminhão** e cada veículo pode fazer **até 2 viagens / semana**:

| Tipo | Capacidade (cx) | Aluguel (R\$/sem) | R\$/km |
|---|---:|---:|---:|
| Pequeno | 60 | 1.500 | 3 |
| Médio | 100 | 2.000 | 4 |
| Grande | 180 | 2.800 | 5 |

### Generalização do modelo

- $y_{jt} \in \{0,1\}$: caminhão $j$ do tipo $t$ alugado?
- $x_{ijtk} \in \{0,1\}$: bar $i$ atendido pelo caminhão $j$ do tipo $t$ na viagem $k \in \{1, 2\}$
- $L_{jtk} \ge 0$: maior distância da viagem $k$ do caminhão $(j,t)$

**Cobertura:** cada bar em exatamente uma combinação $(j, t, k)$.
**Capacidade:** carga da viagem $k$ ≤ $Q_t \cdot y_{jt}$.
**FO:** soma dos aluguéis + soma dos km × R\$/km por tipo.

Implemente abaixo e compare com o resultado homogêneo:

In [ ]:
TIPOS = {
    'P': {'Q': 60,  'rent': 1500, 'ckm': 3},
    'M': {'Q': 100, 'rent': 2000, 'ckm': 4},
    'G': {'Q': 180, 'rent': 2800, 'ckm': 5},
}
MAX_PER_TYPE = 4  # até 4 caminhões de cada tipo disponíveis
VIAGENS = [1, 2]   # cada caminhão pode fazer 1 ou 2 viagens

def solve_fleet_heterogeneo(bares, tipos, max_per_type, viagens):
    solver = pywraplp.Solver.CreateSolver('CBC')
    n = len(bares)
    I = range(n)
    d = bares['d'].tolist()
    dist = bares['dist'].tolist()

    # Index sets
    J = range(max_per_type)
    T = list(tipos.keys())
    K = viagens

    # y[j,t]: caminhão j do tipo t alugado?
    y = {(j, t): solver.IntVar(0, 1, f'y[{j},{t}]') for j in J for t in T}
    # x[i,j,t,k]: bar i atendido na viagem k do caminhão (j,t)?
    x = {(i, j, t, k): solver.IntVar(0, 1, f'x[{i},{j},{t},{k}]')
         for i in I for j in J for t in T for k in K}
    # L[j,t,k]: maior distância da viagem k do caminhão (j,t)
    L = {(j, t, k): solver.NumVar(0, solver.infinity(), f'L[{j},{t},{k}]')
         for j in J for t in T for k in K}

    # Cobertura: cada bar em exatamente 1 (j,t,k)
    for i in I:
        solver.Add(sum(x[i, j, t, k] for j in J for t in T for k in K) == 1)

    # Capacidade por viagem: carga ≤ Q_t · y[j,t]
    for j in J:
        for t in T:
            for k in K:
                solver.Add(sum(d[i] * x[i, j, t, k] for i in I) <= tipos[t]['Q'] * y[j, t])

    # L[j,t,k] ≥ dist[i] · x[i,j,t,k]
    for j in J:
        for t in T:
            for k in K:
                for i in I:
                    solver.Add(L[j, t, k] >= dist[i] * x[i, j, t, k])

    # FO: aluguel total + soma das viagens × 2 · R$/km do tipo
    solver.Minimize(
        sum(tipos[t]['rent'] * y[j, t] for j in J for t in T)
        + sum(2 * tipos[t]['ckm'] * L[j, t, k] for j in J for t in T for k in K)
    )

    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        print(f'Status não-ótimo: {status}')
        return None

    rotas = []
    for j in J:
        for t in T:
            if y[j, t].solution_value() < 0.5:
                continue
            for k in K:
                atendidos = [bares['bar'][i] for i in I if x[i, j, t, k].solution_value() > 0.5]
                if not atendidos:
                    continue
                carga = sum(d[i] for i in I if x[i, j, t, k].solution_value() > 0.5)
                rotas.append({
                    'caminhão': f'{t}{j+1}',
                    'tipo': t,
                    'viagem': k,
                    'bares': ', '.join(atendidos),
                    'caixas': carga,
                    'maior dist': L[j, t, k].solution_value(),
                })
    return {
        'custo_total': solver.Objective().Value(),
        'rotas': pd.DataFrame(rotas),
    }

sol_h = solve_fleet_heterogeneo(bares, TIPOS, MAX_PER_TYPE, VIAGENS)
print(f"Custo total: R$ {sol_h['custo_total']:,.2f}")
print()
print(sol_h['rotas'].to_string(index=False))

print(f"\nComparação:")
print(f"  Homogêneo:    R$ {sol['custo_total']:,.2f}")
print(f"  Heterogêneo:  R$ {sol_h['custo_total']:,.2f}")
print(f"  Economia:     R$ {sol['custo_total'] - sol_h['custo_total']:,.2f}")

### Heterogêneo em Gurobi

Mesma generalização (3 tipos × 2 viagens), agora em `gurobipy`. Note como variáveis multi-indexadas ficam mais limpas:

In [ ]:
def solve_fleet_heterogeneo_gurobi(bares, tipos, max_per_type, viagens):
    m = gp.Model('fleet_hetero')
    m.Params.OutputFlag = 0

    n = len(bares)
    I = range(n)
    d = bares['d'].tolist(); dist = bares['dist'].tolist()
    J = range(max_per_type); T = list(tipos.keys()); K = viagens

    # Variáveis multi-indexadas (tupledict)
    y = m.addVars(J, T, vtype=GRB.BINARY, name='y')
    x = m.addVars(I, J, T, K, vtype=GRB.BINARY, name='x')
    L = m.addVars(J, T, K, vtype=GRB.CONTINUOUS, name='L', lb=0)

    # Cobertura
    m.addConstrs((gp.quicksum(x[i, j, t, k] for j in J for t in T for k in K) == 1
                  for i in I), name='cob')

    # Capacidade por viagem
    m.addConstrs((gp.quicksum(d[i] * x[i, j, t, k] for i in I) <= tipos[t]['Q'] * y[j, t]
                  for j in J for t in T for k in K), name='cap')

    # Distância
    m.addConstrs((L[j, t, k] >= dist[i] * x[i, j, t, k]
                  for i in I for j in J for t in T for k in K), name='dist')

    # FO
    m.setObjective(
        gp.quicksum(tipos[t]['rent'] * y[j, t] for j in J for t in T)
        + gp.quicksum(2 * tipos[t]['ckm'] * L[j, t, k] for j in J for t in T for k in K),
        GRB.MINIMIZE
    )

    m.optimize()
    if m.Status != GRB.OPTIMAL:
        return None

    rotas = []
    for j in J:
        for t in T:
            if y[j, t].X < 0.5: continue
            for k in K:
                atendidos = [bares['bar'][i] for i in I if x[i, j, t, k].X > 0.5]
                if not atendidos: continue
                rotas.append({
                    'caminhão': f'{t}{j+1}', 'tipo': t, 'viagem': k,
                    'bares': ', '.join(atendidos),
                    'caixas': sum(d[i] for i in I if x[i, j, t, k].X > 0.5),
                    'maior dist': L[j, t, k].X,
                })
    return {'custo_total': m.ObjVal, 'rotas': pd.DataFrame(rotas), 'tempo': m.Runtime}

sol_hg = solve_fleet_heterogeneo_gurobi(bares, TIPOS, MAX_PER_TYPE, VIAGENS)
print(f"Custo total (Gurobi heterogêneo): R$ {sol_hg['custo_total']:,.2f}")
print(f"Tempo Gurobi: {sol_hg['tempo']*1000:.1f} ms")
print()
print(sol_hg['rotas'].to_string(index=False))

print(f"\n✓ Mesmo ótimo? OR-Tools R$ {sol_h['custo_total']:,.2f}  vs  Gurobi R$ {sol_hg['custo_total']:,.2f}")

### Para discutir

1. **Qual mix de tipos o ótimo escolheu?** Compare custo fixo vs ganho de capacidade.
2. **Permitir 2 viagens muda o resultado?** Compare com a versão homogênea (1 viagem só).
3. **Sensibilidade — TODO:** rode novamente baixando o aluguel do tipo Grande para R\$ 2.400. A frota muda? A qual valor de aluguel o Grande passa a dominar?

_(Resposta da sensibilidade ficará como exercício para casa.)_

In [ ]:
# TODO: ajustar TIPOS['G']['rent'] = 2400 e rodar novamente
# tipos_v2 = dict(TIPOS)
# tipos_v2['G'] = dict(TIPOS['G'], rent=2400)
# sol_v2 = solve_fleet_heterogeneo(bares, tipos_v2, MAX_PER_TYPE, VIAGENS)
# print(sol_v2['rotas'])

---

## Exercício de extensão 2: CVRP completo — relaxando a aproximação de rota-estrela

Até aqui tratamos cada "rota" como **rota-estrela**: o caminhão ia direto ao bar mais distante e voltava, ignorando que precisa passar pelos outros bares do cluster. Isso subestima a distância real.

**CVRP (Capacitated Vehicle Routing Problem)** relaxa essa simplificação: o solver decide **a ordem real de visita** dentro de cada rota, otimizando a distância percorrida em todo o trajeto.

### O que muda:
- Antes: precisávamos só da **distância do CD ao bar**
- Agora: precisamos da **matriz completa** $d_{ij}$ — distância entre qualquer par de pontos (CD + 8 bares = 9 nós, matriz 9×9)

Vamos comparar 3 abordagens — uma com OR-Tools, duas com Gurobi:
1. **OR-Tools `RoutingModel`** — módulo dedicado a roteamento, com heurísticas built-in
2. **Gurobi MTZ** (Miller-Tucker-Zemlin) — formulação MILP polinomial, sem callbacks
3. **Gurobi com lazy constraints (DFJ)** — *killer feature*: subtours eliminados sob demanda durante o B&B

### Setup: matriz de distâncias 9×9

Coordenadas aproximadas (latitude, longitude) dos pontos em São Paulo, distância calculada com aproximação esférica (1° latitude ≈ 111 km):

In [ ]:
import math

COORDS = {
    'CD':            (-23.567, -46.685),
    'Centro':        (-23.553, -46.635),
    'Pinheiros':     (-23.565, -46.685),
    'Vila Madalena': (-23.555, -46.692),
    'Moema':         (-23.605, -46.665),
    'Tatuapé':       (-23.539, -46.572),
    'Lapa':          (-23.521, -46.706),
    'Itaim':         (-23.583, -46.671),
    'Brooklin':      (-23.612, -46.690),
}
NODES = list(COORDS.keys())
N = len(NODES)
DEPOT = 0
CUSTOMERS = list(range(1, N))

def km(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

# Matriz de distâncias km
D = {(i, j): km(COORDS[NODES[i]], COORDS[NODES[j]]) for i in range(N) for j in range(N)}

# Demanda (re-aproveita do caso simples)
DEM = [0] + bares['d'].tolist()  # 0 para o depot

# Mostra a matriz
import pandas as pd
M = pd.DataFrame(
    [[D[i,j] for j in range(N)] for i in range(N)],
    index=NODES, columns=NODES
).round(1)
print('Matriz de distâncias (km):')
M

### 1) OR-Tools `RoutingModel` — solver dedicado a VRP

OR-Tools tem um **módulo de routing** separado, com heurísticas (PATH_CHEAPEST_ARC, SAVINGS) + busca local (Guided Local Search). Sintaxe diferente do `linear_solver` (pywraplp):

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import time

def solve_cvrp_ortools(D, DEM, Q, max_veh, fixed_cost_decimo=5000, time_limit=10):
    # OR-Tools quer distâncias em inteiros — usamos décimos de km
    Dint = [[int(round(D[i,j] * 10)) for j in range(N)] for i in range(N)]

    mgr = pywrapcp.RoutingIndexManager(N, max_veh, DEPOT)
    rt = pywrapcp.RoutingModel(mgr)

    # Callback de distância
    def dist_cb(from_idx, to_idx):
        return Dint[mgr.IndexToNode(from_idx)][mgr.IndexToNode(to_idx)]
    transit = rt.RegisterTransitCallback(dist_cb)
    rt.SetArcCostEvaluatorOfAllVehicles(transit)

    # Callback de demanda (com dimensão de capacidade)
    def demand_cb(from_idx):
        return DEM[mgr.IndexToNode(from_idx)]
    demand_idx = rt.RegisterUnaryTransitCallback(demand_cb)
    rt.AddDimensionWithVehicleCapacity(demand_idx, 0, [Q]*max_veh, True, 'Capacity')

    # Custo fixo por veículo usado (R$ 2.000 = 5.000 décimos de unidade da FO)
    for v in range(max_veh):
        rt.SetFixedCostOfVehicle(fixed_cost_decimo, v)

    # Search params
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.seconds = time_limit

    t0 = time.time()
    sol = rt.SolveWithParameters(params)
    t = time.time() - t0
    if sol is None:
        return None

    rotas = []
    total_km = 0
    for v in range(max_veh):
        idx = rt.Start(v)
        if rt.IsEnd(sol.Value(rt.NextVar(idx))):
            continue
        seq = []
        while not rt.IsEnd(idx):
            seq.append(NODES[mgr.IndexToNode(idx)])
            idx = sol.Value(rt.NextVar(idx))
        seq.append(NODES[mgr.IndexToNode(idx)])
        d_rota = sum(D[NODES.index(seq[i]), NODES.index(seq[i+1])] for i in range(len(seq)-1))
        rotas.append({'rota': ' → '.join(seq),
                      'caixas': sum(DEM[NODES.index(n)] for n in seq),
                      'km': d_rota})
        total_km += d_rota
    return {'rotas': rotas, 'total_km': total_km,
            'n_veic': len(rotas),
            'custo': 2000*len(rotas) + 4*total_km,
            'tempo_ms': t*1000}

sol_ortools = solve_cvrp_ortools(D, DEM, Q, MAX_TRUCKS)
print(f"OR-Tools RoutingModel:  custo = R$ {sol_ortools['custo']:,.2f}  ({sol_ortools['n_veic']} veículos, {sol_ortools['total_km']:.1f} km, {sol_ortools['tempo_ms']:.0f} ms)")
for r in sol_ortools['rotas']:
    print(f"  {r['rota']:<60} {r['caixas']:>3} cx, {r['km']:.1f} km")

### 2) Gurobi MTZ — formulação MILP polinomial

Variáveis:
- $x_{ijk} \in \{0,1\}$: veículo $k$ usa o arco $i \to j$
- $y_k \in \{0,1\}$: veículo $k$ é usado
- $u_{ik} \ge 1$: ordem de visita do cliente $i$ pelo veículo $k$ (MTZ)

Restrição-chave de **eliminação de subtour (MTZ)**: para cada par $(i, j)$ de clientes e cada veículo $k$:
$$u_{ik} - u_{jk} + (n-1)\,x_{ijk} \le n - 2$$

Lê-se: se $x_{ijk} = 1$ (veículo $k$ vai de $i$ para $j$), então $u_{jk} \ge u_{ik} + 1$ — a ordem cresce.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_cvrp_gurobi_mtz(D, DEM, Q, max_veh):
    m = gp.Model('cvrp_mtz')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = 30

    x = m.addVars(range(N), range(N), range(max_veh), vtype=GRB.BINARY, name='x')
    y = m.addVars(range(max_veh), vtype=GRB.BINARY, name='y')
    u = m.addVars(CUSTOMERS, range(max_veh), lb=1, ub=N-1, name='u')

    # Cada cliente visitado 1 vez
    m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(max_veh)) == 1
                  for j in CUSTOMERS))
    # Conservação de fluxo
    m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
                  gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
                  for j in range(N) for k in range(max_veh)))
    # Veículo sai do depot no máximo 1 vez (= y[k])
    m.addConstrs((gp.quicksum(x[DEPOT,j,k] for j in CUSTOMERS) == y[k] for k in range(max_veh)))
    # Capacidade
    m.addConstrs((gp.quicksum(DEM[j]*x[i,j,k] for i in range(N) for j in CUSTOMERS if i!=j) <= Q*y[k]
                  for k in range(max_veh)))
    # Sem self-loops
    m.addConstrs((x[i,i,k] == 0 for i in range(N) for k in range(max_veh)))
    # MTZ subtour elimination
    for k in range(max_veh):
        for i in CUSTOMERS:
            for j in CUSTOMERS:
                if i != j:
                    m.addConstr(u[i,k] - u[j,k] + (N-1)*x[i,j,k] <= N-2)

    m.setObjective(
        2000*gp.quicksum(y[k] for k in range(max_veh))
        + 4*gp.quicksum(D[i,j]*x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(max_veh)),
        GRB.MINIMIZE
    )

    t0 = time.time()
    m.optimize()
    t = time.time() - t0

    # Decodificar rotas
    rotas = []
    for k in range(max_veh):
        if y[k].X < 0.5: continue
        cur = DEPOT; seq = [NODES[cur]]
        while True:
            prox = next((j for j in range(N) if j!=cur and x[cur,j,k].X > 0.5), None)
            if prox is None or prox == DEPOT:
                seq.append('CD'); break
            seq.append(NODES[prox]); cur = prox
        d_rota = sum(D[NODES.index(seq[i]), NODES.index(seq[i+1])] for i in range(len(seq)-1))
        rotas.append({'rota': ' → '.join(seq),
                      'caixas': sum(DEM[NODES.index(n)] for n in seq),
                      'km': d_rota})
    return {'rotas': rotas, 'custo': m.ObjVal, 'tempo_ms': t*1000}

sol_mtz = solve_cvrp_gurobi_mtz(D, DEM, Q, MAX_TRUCKS)
print(f"Gurobi MTZ:  custo = R$ {sol_mtz['custo']:,.2f}  ({sol_mtz['tempo_ms']:.0f} ms)")
for r in sol_mtz['rotas']:
    print(f"  {r['rota']:<60} {r['caixas']:>3} cx, {r['km']:.1f} km")

### 3) Gurobi com lazy constraints (DFJ + callbacks) — o killer feature

**Dantzig-Fulkerson-Johnson (DFJ)** elimina subtour de outra forma: para qualquer subconjunto $S$ de clientes,
$$\sum_{i,j \in S,\ i \ne j} x_{ijk} \le |S| - 1$$

Problema: o **número de subconjuntos é exponencial** ($2^n$). Não dá pra adicionar todas as restrições upfront.

**Solução elegante (só em solvers comerciais):** *callbacks*. O solver Gurobi nos dá um *hook* dentro do B&B — toda vez que ele encontra uma solução inteira, chamamos uma função Python que:
1. Procura subtour na solução corrente
2. Se encontrar, adiciona a constraint específica como **lazy constraint**
3. Continua o B&B

É uma das razões clássicas para investir em Gurobi/CPLEX em problemas de roteamento de larga escala — solvers open-source com suporte a callbacks são raros.

In [ ]:
def solve_cvrp_gurobi_callbacks(D, DEM, Q, max_veh):
    m = gp.Model('cvrp_dfj_lazy')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = 30
    m.Params.LazyConstraints = 1   # OBRIGATÓRIO para callbacks lazy

    x = m.addVars(range(N), range(N), range(max_veh), vtype=GRB.BINARY, name='x')
    y = m.addVars(range(max_veh), vtype=GRB.BINARY, name='y')
    m._x = x  # disponível dentro do callback

    # Sem MTZ — vamos confiar em lazy constraints
    m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j for k in range(max_veh)) == 1
                  for j in CUSTOMERS))
    m.addConstrs((gp.quicksum(x[i,j,k] for i in range(N) if i!=j) ==
                  gp.quicksum(x[j,i,k] for i in range(N) if i!=j)
                  for j in range(N) for k in range(max_veh)))
    m.addConstrs((gp.quicksum(x[DEPOT,j,k] for j in CUSTOMERS) == y[k] for k in range(max_veh)))
    m.addConstrs((gp.quicksum(DEM[j]*x[i,j,k] for i in range(N) for j in CUSTOMERS if i!=j) <= Q*y[k]
                  for k in range(max_veh)))
    m.addConstrs((x[i,i,k] == 0 for i in range(N) for k in range(max_veh)))

    m.setObjective(
        2000*gp.quicksum(y[k] for k in range(max_veh))
        + 4*gp.quicksum(D[i,j]*x[i,j,k] for i in range(N) for j in range(N) if i!=j for k in range(max_veh)),
        GRB.MINIMIZE
    )

    def callback(model, where):
        """Chamado pelo Gurobi durante o B&B. Detecta subtours e adiciona constraints."""
        if where != GRB.Callback.MIPSOL:
            return
        for k in range(max_veh):
            # Recupera valores da solução corrente para o veículo k
            vals = model.cbGetSolution([model._x[i,j,k] for i in range(N) for j in range(N) if i!=j])
            arcs = [(i,j) for idx,(i,j) in enumerate([(i,j) for i in range(N) for j in range(N) if i!=j]) if vals[idx] > 0.5]
            if not arcs: continue

            # BFS a partir do depot — quem está alcançável é parte da 'rota legítima'
            visitados = {DEPOT}; stack = [DEPOT]
            while stack:
                n = stack.pop()
                for (i,j) in arcs:
                    if i == n and j not in visitados:
                        visitados.add(j); stack.append(j)
            # Nós em arcs mas não alcançáveis = subtour
            todos = {i for (i,_) in arcs} | {j for (_,j) in arcs}
            subtour = todos - visitados
            if subtour:
                S = list(subtour)
                model.cbLazy(
                    gp.quicksum(model._x[i,j,k] for i in S for j in S if i!=j) <= len(S) - 1
                )

    t0 = time.time()
    m.optimize(callback)
    t = time.time() - t0

    rotas = []
    for k in range(max_veh):
        if y[k].X < 0.5: continue
        cur = DEPOT; seq = [NODES[cur]]
        while True:
            prox = next((j for j in range(N) if j!=cur and x[cur,j,k].X > 0.5), None)
            if prox is None or prox == DEPOT:
                seq.append('CD'); break
            seq.append(NODES[prox]); cur = prox
        d_rota = sum(D[NODES.index(seq[i]), NODES.index(seq[i+1])] for i in range(len(seq)-1))
        rotas.append({'rota': ' → '.join(seq),
                      'caixas': sum(DEM[NODES.index(n)] for n in seq),
                      'km': d_rota})
    return {'rotas': rotas, 'custo': m.ObjVal, 'tempo_ms': t*1000}

sol_dfj = solve_cvrp_gurobi_callbacks(D, DEM, Q, MAX_TRUCKS)
print(f"Gurobi DFJ+callbacks:  custo = R$ {sol_dfj['custo']:,.2f}  ({sol_dfj['tempo_ms']:.0f} ms)")
for r in sol_dfj['rotas']:
    print(f"  {r['rota']:<60} {r['caixas']:>3} cx, {r['km']:.1f} km")

### Comparação das três abordagens

Vamos sumarizar:

In [ ]:
comparacao = pd.DataFrame([
    {'método': 'Rota-estrela (caso simples)',     'custo': sol['custo_total'],         'tempo (ms)': '—'},
    {'método': 'OR-Tools RoutingModel',           'custo': sol_ortools['custo'],       'tempo (ms)': f"{sol_ortools['tempo_ms']:.0f}"},
    {'método': 'Gurobi MTZ (MILP exato)',         'custo': sol_mtz['custo'],           'tempo (ms)': f"{sol_mtz['tempo_ms']:.0f}"},
    {'método': 'Gurobi DFJ + lazy constraints',   'custo': sol_dfj['custo'],           'tempo (ms)': f"{sol_dfj['tempo_ms']:.0f}"},
])
comparacao['custo'] = comparacao['custo'].apply(lambda v: f'R$ {v:,.2f}')
print(comparacao.to_string(index=False))

### Lições deste módulo

1. **A rota-estrela subestima o custo real.** Para o caso Beerlink, o custo real do plano (R\$ 8.250) é R\$ 49 maior do que a estimativa otimista (R\$ 8.201). Em problemas maiores essa subestimação pode ser de 10–30 %.

2. **Para problemas pequenos, qualquer solver serve.** OR-Tools, Gurobi MTZ e Gurobi DFJ convergem para a mesma resposta. Escolha pela ergonomia do código.

3. **Em problemas reais (50+ clientes), CVRP fica difícil.** É aí que a estratégia muda:
   - **OR-Tools RoutingModel** continua ótimo para problemas até centenas de clientes — heurísticas + busca local
   - **Gurobi + lazy constraints** é o padrão da indústria para CVRP exato com 50–100 clientes — separação de subtour just-in-time é fundamental
   - **Heurísticas dedicadas** (LKH, HGS, etc.) vão além para milhares de nós

4. **Lazy constraints é um recurso pago.** Solvers open-source (CBC, GLPK) não oferecem, ou oferecem de forma muito limitada. Este é um dos casos onde a licença Gurobi se paga em projetos de roteamento.